# PS3 - Problem 5 - K-means for compression

### (a)

The following cells implement the code for the K-means algorithm. To show result image, run all the cells in the notebook.

In [ ]:
# Loading libraries for following steps
import numpy as np
import os
import matplotlib.pyplot as plt
from matplotlib.image import imread

##### Show large image

In [ ]:
src_path = os.path.join('data', 'peppers-large.tiff')
A = imread(src_path)
plt.imshow(A);

##### Show small image

In [ ]:
src_path = os.path.join('data', 'peppers-small.tiff')
B = imread(src_path)
plt.imshow(B);

##### Vector quantization through K-means

In [ ]:
np.random.seed(0)

# Variables and parameters
K = 16          # Number of clusters
it = 0          # Current iteration
J = None        # Value of the distortion function
eps = 1e-3      # Epsilon for convergence

# Convert image into a design matrix (m x 3) [m is the number of pixels]
x = B.reshape(-1,3)
m = x.shape[0]

# Initialize centroids
indices = np.random.choice(np.arange(m), K, replace=False)
centroids = x[indices].astype('float64')

# Function to compute distance between points and centroids
def compute_distance(x, centroids):
    return np.linalg.norm(
        x[:, np.newaxis, :] - centroids[np.newaxis, :, :],
        axis=2
    )
dist_sq = compute_distance(x, centroids)

# K-means iteration
while it < 300:
    # 1. Assign points to centroids
    z = np.argmin(dist_sq, axis=1)
    # 2. Re-compute centroids and update distortion function
    for i in range(K):
        points = x[z==i]    # Find points for the current centroid
        # If no point is associated to the centroid, continue to next centroid
        if len(points) > 0:
            centroids[i] = np.mean(points, axis=0)
    # 3. Re-compute distances from centroids and update distortion function
    dist_sq = compute_distance(x,centroids)
    old_J = J
    J = np.mean(dist_sq[np.arange(m),z])
    # 4. Check convergence and potentially break
    if old_J is not None:
        diff = J - old_J
        if np.isfinite(diff) and abs(diff) < eps:
            break
    # Update number of iterations
    it += 1

print(f"Number of iterations: {it}")

# Plot
fig, ax = plt.subplots(1,2, figsize=(10,5))
# Small image
ax[0].imshow(B)
ax[0].set_title("Small image")
# Debug image, plot cluster numbers as a picture 
z_B = z.reshape(B.shape[:-1])
ax[1].imshow(z_B);
ax[1].set_title("Clusters")
plt.show();

##### Image compression

In [ ]:
# Duplicate image
C = A.copy().reshape(-1,3)
# Cluster pixels
dist_sq = compute_distance(C,centroids)     # Compute distances
z = np.argmin(dist_sq, axis=1)              # Assign to cluster
z = z.reshape(A.shape[:-1])
# Re-create final image
C = np.zeros(A.shape, dtype=int)
for i in range(len(C)):
    for j in range(len(C)):
        C[i,j,:] = centroids[z[i,j]]
# Plot
fig, ax = plt.subplots(1,2,figsize=(10,5))
ax[0].imshow(A)
ax[0].set_title("Large image, original")
ax[1].imshow(C)
ax[1].set_title("Large image, compressed")
plt.show();

### (b)

If any color in the image can be represented with one of 16 values, then it is possible to represent the whole image as if it was composed of a single channel with values between 0 and 15. In this case, assuming each of the numbers required 1 byte to be represented, the image would require $512 \times 512 = 262,144$ bytes to encode the pixels, and an additional $16 \times 3 = 48$ bytes to encode information about which color each number represents. In that case, the main gain in image compression would be due to reducing the 3 channels into a single one, thus compressing the image by a factor of 3. 

However, the first 16 numbers can be represented with less than 1 byte: namely, only 4 bits (half a byte) are needed. This implies that each pixel could be represented by half of the bytes from the previous case, and the image could be further compressed by a factor of 2, thus attaining an overall compression factor of 6.  